<a href="https://colab.research.google.com/github/AICHUCKY/Ai-with-Chucky-Colab-Notebooks/blob/main/ComfyUI_Optimized_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ ComfyUI Cloud Studio

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

### ✨ **Key Features:**
- 🚀 **Instant Canvas Boot:** Optimized to load your node graph and canvas in seconds without UI freeze or infinite loading.
- 📥 **Smart Model & Node Downloader:** Download Checkpoints, GGUF/UNets, LoRAs, VAEs, and custom nodes directly from Hugging Face and Civitai.
- 🧩 **ComfyUI-Manager Included:** Browse, install, and update your favorite custom nodes and workflows right inside the UI.
- 🌐 **5 Remote Access Options:** Connect easily through Cloudflare, Colab Native URL, Localtunnel, Pinggy, or Ngrok.
- 💾 **Google Drive Persistence & Workspace Backup:** Save your downloaded models, outputs, and custom nodes directly to Drive so you never lose your setup.
- 🔊 **Anti-Disconnect Keep-Alive:** Integrated browser activity guard to keep your Colab session from disconnecting while you create.

In [ ]:
#@title 1. Initialize ComfyUI Environment
#@markdown Sets up ComfyUI, ComfyUI-Manager, dependencies, and fast-boot optimizations.

import os
import re
import shutil
import subprocess
import sys
from google.colab import drive
from IPython.display import HTML, display

display(HTML('<div style="background: linear-gradient(90deg, #4b6cb7 0%, #182848 100%); color: white; padding: 12px; border-radius: 6px; font-weight: bold; text-align: center;">⚡ Initializing ComfyUI Environment...</div>'))

# --- Configuration ---
MOUNT_DRIVE = False #@param {type:"boolean"}
UPDATE_COMFY_UI = False #@param {type:"boolean"}
FORCE_CLEAN_CACHE = False #@param {type:"boolean"}

LOCAL_WORKSPACE = "/content/ComfyUI"
DRIVE_WORKSPACE = "/content/drive/MyDrive/ComfyUI"
CACHE_TAR = os.path.join(DRIVE_WORKSPACE, "comfy_ui_cache.tar")

def stream_cmd(cmd, cwd=None):
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd, bufsize=1)
    for line in iter(process.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()
    process.wait()

print("\n📦 [1/4] Installing UV, aria2c, pigz, and cloudflared...")
subprocess.run(["pip", "install", "-q", "uv"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "-qq", "aria2", "pigz"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/usr/local/bin/cloudflared"])
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"])

if MOUNT_DRIVE:
    print("\n💾 Requesting Google Drive Access...")
    drive.mount('/content/drive')

print("\n📂 [2/4] Setting up ComfyUI Core...")
if not os.path.exists(LOCAL_WORKSPACE):
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI", LOCAL_WORKSPACE])
else:
    if UPDATE_COMFY_UI:
        print("   🔄 Pulling latest core updates...")
        subprocess.run(["git", "pull"], cwd=LOCAL_WORKSPACE)

if MOUNT_DRIVE:
    print("   🔗 Linking model storage directories to Google Drive...")
    for d in ["models", "output", "input"]:
        local_path = os.path.join(LOCAL_WORKSPACE, d)
        drive_path = os.path.join(DRIVE_WORKSPACE, d)
        os.makedirs(drive_path, exist_ok=True)
        if os.path.exists(local_path) and not os.path.islink(local_path):
            shutil.rmtree(local_path)
        if not os.path.exists(local_path):
            os.symlink(drive_path, local_path)

    if FORCE_CLEAN_CACHE and os.path.exists(CACHE_TAR):
        print("   🗑️ Removing old cache tarball...")
        os.remove(CACHE_TAR)

    if os.path.exists(CACHE_TAR):
        print("   📦 Unpacking cached workspace...")
        os.system(f"tar -I pigz -xf '{CACHE_TAR}' -C '{LOCAL_WORKSPACE}' > /dev/null 2>&1")

os.makedirs(os.path.join(LOCAL_WORKSPACE, "custom_nodes"), exist_ok=True)
os.makedirs(os.path.join(LOCAL_WORKSPACE, "user"), exist_ok=True)

# Install ComfyUI-Manager
manager_path = os.path.join(LOCAL_WORKSPACE, "custom_nodes", "ComfyUI-Manager")
if not os.path.exists(os.path.join(manager_path, "__init__.py")):
    if os.path.exists(manager_path):
        shutil.rmtree(manager_path)
    print("\n📦 [3/4] Installing ComfyUI-Manager...")
    subprocess.run(["git", "clone", "https://github.com/Comfy-Org/ComfyUI-Manager.git", manager_path])

# Non-blocking lazy config for ComfyUI-Manager
mgr_config = """[default]
network_mode = standard
preview_method = auto
badge_mode = None
security_level = normal
migrated = true
skip_update_check = true
component_policy = remote
lazy_load = true
"""
for conf_dir in [
    os.path.join(LOCAL_WORKSPACE, "user", "default", "ComfyUI-Manager"),
    os.path.join(LOCAL_WORKSPACE, "user", "ComfyUI-Manager"),
    os.path.join(LOCAL_WORKSPACE, "user", "__manager"),
    manager_path
]:
    os.makedirs(conf_dir, exist_ok=True)
    with open(os.path.join(conf_dir, "config.ini"), "w") as f:
        f.write(mgr_config)

print("\n🛠️ [4/4] Installing Python dependencies via UV...")
stream_cmd(["uv", "pip", "install", "--system", "-r", "requirements.txt"], cwd=LOCAL_WORKSPACE)

stream_cmd(["uv", "pip", "install", "--system",
            "av", "imageio-ffmpeg", "torchsde", "einops", "transformers", "safetensors",
            "kornia", "spandrel", "scipy", "soundfile", "insightface", "onnxruntime-gpu",
            "alembic", "blake3", "comfy-kitchen", "comfy-aimdo", "comfy-angle", "gguf", "piexif",
            "comfyui-frontend-package", "pydantic", "pydantic-settings", "simpleeval", "accelerate"])

mgr_req = os.path.join(manager_path, "requirements.txt")
if os.path.exists(mgr_req):
    stream_cmd(["uv", "pip", "install", "--system", "-r", mgr_req])

# --- GZIP TURBO PATCH ---
server_file = os.path.join(LOCAL_WORKSPACE, "server.py")
if os.path.exists(server_file):
    with open(server_file, "r") as f:
        content = f.read()
    if "gzip_object_info" not in content:
        patch = """
import gzip as _gzip
import json as _json

async def gzip_object_info(self, request):
    data = _json.dumps(self.node_info()).encode('utf-8')
    accept = request.headers.get('Accept-Encoding', '')
    if 'gzip' in accept:
        return web.Response(body=_gzip.compress(data), content_type='application/json', headers={'Content-Encoding': 'gzip'})
    return web.Response(body=data, content_type='application/json')
PromptServer.get_object_info = gzip_object_info
"""
        with open(server_file, "a") as f:
            f.write(patch)

print("\n✅ [SYSTEM READY] ComfyUI environment initialized.")

In [ ]:
#@title 2. Smart Model & Node Downloader (16-Stream aria2c)
#@markdown Paste direct model or git URLs (supports multi-line inputs, Hugging Face, and Civitai).

import os
import re
import shutil
import subprocess
import urllib.parse
import requests

WORKSPACE = "/content/ComfyUI"
HF_TOKEN = "" #@param {type:"string"}
CIVITAI_API_KEY = "" #@param {type:"string"}

CHECKPOINT_URLS = "" #@param {type:"string"}
UNET_DIFFUSION_URLS = "" #@param {type:"string"}
TEXT_ENCODER_URLS = "" #@param {type:"string"}
CLIP_VISION_URLS = "" #@param {type:"string"}
VAE_URLS = "" #@param {type:"string"}
LORA_URLS = "" #@param {type:"string"}
CONTROLNET_URLS = "" #@param {type:"string"}
UPSCALE_MODELS_URLS = "" #@param {type:"string"}
LATENT_UPSCALE_URLS = "" #@param {type:"string"}
EMBEDDING_URLS = "" #@param {type:"string"}
CUSTOM_NODE_URLS = "" #@param {type:"string"}

DIRS = {
    "checkpoints":           os.path.join(WORKSPACE, "models/checkpoints"),
    "unet":                  os.path.join(WORKSPACE, "models/unet"),
    "clip":                  os.path.join(WORKSPACE, "models/clip"),
    "clip_vision":           os.path.join(WORKSPACE, "models/clip_vision"),
    "vae":                   os.path.join(WORKSPACE, "models/vae"),
    "loras":                 os.path.join(WORKSPACE, "models/loras"),
    "controlnet":            os.path.join(WORKSPACE, "models/controlnet"),
    "upscale_models":        os.path.join(WORKSPACE, "models/upscale_models"),
    "latent_upscale_models": os.path.join(WORKSPACE, "models/latent_upscale_models"),
    "embeddings":            os.path.join(WORKSPACE, "models/embeddings"),
    "custom_nodes":          os.path.join(WORKSPACE, "custom_nodes")
}

def clean_url(url):
    url = url.strip()
    if "huggingface.co" in url and "/blob/" in url:
        url = url.replace("/blob/", "/resolve/")
    if "civitai.com" in url and CIVITAI_API_KEY.strip():
        sep = "&" if "?" in url else "?"
        url = f"{url}{sep}token={CIVITAI_API_KEY.strip()}"
    return url

def resolve_filename(url):
    filename = None
    token = HF_TOKEN.strip()
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        if 'huggingface.co' in url and token:
            headers['Authorization'] = f'Bearer {token}'

        response = requests.head(url, allow_redirects=True, headers=headers, timeout=10)
        cd = response.headers.get('content-disposition', '')
        if cd:
            filenames = re.findall(r'filename\*?=(?:UTF-8\'\')?"?([^;\r\n]+)"?', cd)
            if filenames:
                filename = urllib.parse.unquote(filenames[0].strip())
        if not filename:
            filename = os.path.basename(urllib.parse.urlparse(response.url).path)
    except Exception:
        pass

    if not filename or filename in ['', 'download', 'resolve']:
        filename = os.path.basename(urllib.parse.urlparse(url).path)

    return urllib.parse.unquote(filename)

def download_file_aria2(url, target_dir):
    url = clean_url(url)
    filename = resolve_filename(url)
    token = HF_TOKEN.strip()
    print(f"   📥 Downloading: \033[96m{filename}\033[0m")

    cmd = [
        "aria2c",
        "--continue=true",
        "--max-connection-per-server=16",
        "--split=16",
        "--min-split-size=1M",
        "--file-allocation=none",
        f"--dir={target_dir}",
        f"--out={filename}",
        "--summary-interval=0",
        "--console-log-level=warn"
    ]
    if "huggingface.co" in url and token:
        cmd.append(f"--header=Authorization: Bearer {token}")

    cmd.append(url)
    res = subprocess.run(cmd)
    if res.returncode == 0:
        print(f"      ✅ Saved to: {os.path.join(target_dir, filename)}\n")
    else:
        print(f"      ❌ Download failed: {url}\n")

def process_downloads(urls_str, target_dir, is_node=False):
    if not urls_str.strip(): return
    url_list = [u.strip() for u in urls_str.replace(',', '\n').split('\n') if u.strip()]
    os.makedirs(target_dir, exist_ok=True)

    print(f"\n📁 Category: \033[92m{os.path.basename(target_dir)}\033[0m")
    for raw_url in url_list:
        if is_node:
            node_name = raw_url.split('/')[-1].replace('.git', '')
            node_path = os.path.join(target_dir, node_name)
            if not os.path.exists(node_path):
                print(f"   ⬇️ Cloning Node: {node_name}...")
                subprocess.run(["git", "clone", raw_url, node_path])
                req = os.path.join(node_path, "requirements.txt")
                if os.path.exists(req):
                    print(f"      📦 Installing requirements via UV...")
                    subprocess.run(["uv", "pip", "install", "--system", "-r", req])
                print("      ✅ Node Installed\n")
            else:
                print(f"   ⏩ Node exists: {node_name}\n")
        else:
            download_file_aria2(raw_url, target_dir)

process_downloads(CHECKPOINT_URLS,     DIRS["checkpoints"])
process_downloads(UNET_DIFFUSION_URLS, DIRS["unet"])
process_downloads(TEXT_ENCODER_URLS,   DIRS["clip"])
process_downloads(CLIP_VISION_URLS,    DIRS["clip_vision"])
process_downloads(VAE_URLS,            DIRS["vae"])
process_downloads(LORA_URLS,           DIRS["loras"])
process_downloads(CONTROLNET_URLS,     DIRS["controlnet"])
process_downloads(UPSCALE_MODELS_URLS, DIRS["upscale_models"])
process_downloads(LATENT_UPSCALE_URLS, DIRS["latent_upscale_models"])
process_downloads(EMBEDDING_URLS,      DIRS["embeddings"])
process_downloads(CUSTOM_NODE_URLS,    DIRS["custom_nodes"], is_node=True)

print("🎉 All download tasks finished.")

In [ ]:
#@title 3. Anti-Disconnect Keep-Alive
#@markdown Runs a background audio heartbeat and JavaScript ping to keep the Colab container awake.

from IPython.display import HTML, display

display(HTML('''
<div style="padding: 10px; background: #222; border-left: 4px solid #00d2ff; color: #eee; font-family: sans-serif;">
  <b>🔊 Keep-Alive Protocol Active</b><br>
  <span style="font-size: 12px; color: #aaa;">Browser tab sleep prevention running via web audio loop & activity ping.</span>
  <div style="margin-top: 8px;">
    <audio src="https://raw.githubusercontent.com/anars/blank-audio/master/10-minutes-of-silence.mp3" autoplay loop controls style="height: 30px; width: 250px;"></audio>
  </div>
</div>
<script>
  function keepAlive() {
    console.log("[Keep-Alive] Ping sent: " + new Date().toISOString());
  }
  setInterval(keepAlive, 60000);
</script>
'''))

In [ ]:
#@title 4. Start ComfyUI Session (Fast Canvas Boot)
#@markdown Select Cloudflared (Fastest) or Colab Native URL to bypass Localtunnel lag.

import subprocess
import threading
import time
import socket
import os
import sys
import re
import urllib.request
from IPython.display import HTML, display
from google.colab import output

# --- User Options ---
TUNNEL_PROVIDER = "Localtunnel" #@param ["Cloudflared (Recommended - No Auth)", "Colab Native URL (Direct)", "Localtunnel", "Pinggy (SSH)", "Ngrok"]
NGROK_AUTH_TOKEN = "" #@param {type:"string"}
MEMORY_PROFILE = "Standard (Auto-Detect)" #@param ["Standard (Auto-Detect)", "Low VRAM (T4 GPU / Heavy Models)", "High VRAM (A100 GPU Only)"]

WORKSPACE = "/content/ComfyUI"
PORT = 8188

display(HTML('<div style="background: linear-gradient(90deg, #ff416c 0%, #ff4b2b 100%); color: white; padding: 12px; border-radius: 6px; font-weight: bold; text-align: center;">[SYSTEM] Booting Turbo ComfyUI Engine...</div>'))

ARGS = [
    "--listen", "0.0.0.0",
    "--port", str(PORT),
    "--preview-method", "auto",
    "--enable-cors-header", "*",
    "--disable-auto-launch",
    "--cuda-malloc"
]

if "Low VRAM" in MEMORY_PROFILE:
    ARGS.append("--lowvram")
elif "High VRAM" in MEMORY_PROFILE:
    ARGS.append("--highvram")

def wait_for_port(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        res = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if res == 0:
            break

def render_tunnel_banner(name, url, note=""):
    display(HTML(f'''
    <div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); color: #000; padding: 14px; border-radius: 8px; font-weight: bold; font-size: 15px; margin: 15px 0; text-align: center; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
      🚀 <b>{name} READY:</b> <a href="{url}" target="_blank" style="color: #002b49; text-decoration: underline; margin-left: 8px;">{url}</a>
      {f'<br><span style="font-size: 12px; color: #003b1f;">{note}</span>' if note else ''}
    </div>
    '''))

def run_tunnel(provider, port):
    wait_for_port(port)

    if "Cloudflared" in provider:
        p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate", "--edge-ip-version", "4"],
                             stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        for line in p.stderr:
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                render_tunnel_banner("CLOUDFLARE TURBO TUNNEL", match.group(0))
                break

    elif "Colab Native" in provider:
        colab_url = output.eval_js(f'google.colab.kernel.proxyPort({port})')
        render_tunnel_banner("COLAB DIRECT PROXY", colab_url, "Direct connection inside active Google account")

    elif "Localtunnel" in provider:
        if not os.path.exists("/usr/local/bin/lt"):
            os.system("npm install -g localtunnel > /dev/null 2>&1")
        try:
            ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
        except:
            ip = "Unavailable"
        print(f"\n👉 **Localtunnel Password (IP):** {ip}\n")
        p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        for line in p.stdout:
            if "your url is" in line.lower():
                url = line.split("is:")[-1].strip() if "is:" in line else line.split("is")[-1].strip()
                render_tunnel_banner("LOCALTUNNEL", url, f"Endpoint Password: {ip}")
                break

    elif "Pinggy" in provider:
        p = subprocess.Popen(["ssh", "-o", "StrictHostKeyChecking=no", "-p", "443", f"-R0:localhost:{port}", "a.pinggy.io"],
                             stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        for line in p.stdout:
            match = re.search(r'https://[a-zA-Z0-9-]+\.pinggy\.link', line)
            if match:
                render_tunnel_banner("PINGGY TUNNEL", match.group(0))
                break

    elif "Ngrok" in provider:
        if not NGROK_AUTH_TOKEN.strip():
            print("\n❌ [ERROR] Ngrok selected but NGROK_AUTH_TOKEN parameter is empty!")
            return
        os.system("pip install -q pyngrok")
        from pyngrok import ngrok
        ngrok.set_auth_token(NGROK_AUTH_TOKEN.strip())
        tunnel = ngrok.connect(port, "http")
        render_tunnel_banner("NGROK TUNNEL", tunnel.public_url)

# Launch tunnel watcher in background
threading.Thread(target=run_tunnel, daemon=True, args=(TUNNEL_PROVIDER, PORT)).start()

if os.path.exists(os.path.join(WORKSPACE, "main.py")):
    os.chdir(WORKSPACE)
    print(f"[SYSTEM] Booting ComfyUI Turbo [Profile: {MEMORY_PROFILE}] [Tunnel: {TUNNEL_PROVIDER}]")
    print("[SYSTEM] Streaming Logs...\n" + "-"*60)

    cmd = ["python", "main.py"] + ARGS
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    for line in iter(process.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()
else:
    print("❌ [FATAL ERROR] ComfyUI directory missing.")

In [ ]:
#@title 5. High-Speed Workspace Backup to Drive
#@markdown Archives custom nodes, user workflows, and settings directly to Google Drive using multi-threaded compression.

import os
WORKSPACE = "/content/ComfyUI"
DRIVE_WORKSPACE = "/content/drive/MyDrive/ComfyUI"
CACHE_TAR = os.path.join(DRIVE_WORKSPACE, "comfy_ui_cache.tar")

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount('/content/drive')

os.makedirs(DRIVE_WORKSPACE, exist_ok=True)

if os.path.exists(WORKSPACE):
    print("🔄 Archiving custom nodes & user directory with pigz...")
    os.chdir(WORKSPACE)
    os.system(f"tar -I pigz -cf '{CACHE_TAR}' custom_nodes user")
    print(f"✅ Fast archive saved to Drive: {CACHE_TAR}")
else:
    print("⚠️ Workspace not found.")